# 👀 02 · 图注意力 GAT 与采样邻居 GraphSAGE

> GCN 给所有邻居同等权重——**GAT 用注意力加权**，**GraphSAGE 用采样+可学习聚合**。
> 本节手写**注意力系数（attention score）与加权聚合**，再手写 **GraphSAGE 的邻居采样与聚合**。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | GAT 与 GCN 区别？ | 邻居权重可学习（注意力） |
| 2 | 注意力系数怎么算？ | e = LeakyReLU(a[Wh_i ‖ Wh_j]) |
| 3 | GraphSAGE 怎么采样？ | 固定大小邻居采样 |
| 4 | 聚合函数有哪些？ | Mean / LSTM / Pool |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import networkx as nx
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 构造小图与特征

In [2]:
G = nx.karate_club_graph()
nodes = list(G.nodes())
n = len(nodes)
d = 4
X = rng.normal(0, 1, (n, d))
A = nx.to_numpy_array(G)
print('节点:', n, '特征维度:', d)
print('➡️ GAT 关键差异：给每个邻居算一个可学习权重 α_ij（归一化后和为 1）')

节点: 34 特征维度: 4
➡️ GAT 关键差异：给每个邻居算一个可学习权重 α_ij（归一化后和为 1）


## 2️⃣ 手写注意力系数（单头 GAT 前向）

In [3]:
def gat_forward(X, A, W, a_vec, seed=9):
    r = np.random.default_rng(seed)
    # 线性变换
    H = X @ W                                # (n, d_out)
    # 拼接 (i,j) 与 (j,i) 方向的注意力打分
    def attn_score(h_i, h_j):
        z = np.concatenate([h_i, h_j])
        return float(np.dot(a_vec, z))       # 简化：线性打分

    d_out = W.shape[1]
    H_norm = H / (np.linalg.norm(H, axis=1, keepdims=True) + 1e-9)
    out = np.zeros_like(H)
    nbrs_idx = [list(G.neighbors(i)) for i in nodes]
    for i in range(n):
        nbrs = nbrs_idx[i]
        if not nbrs:
            out[i] = H[i]
            continue
        scores = np.array([attn_score(H_norm[i], H_norm[j]) for j in nbrs])
        alphas = np.exp(scores - scores.max()) / np.exp(scores - scores.max()).sum()
        out[i] = alphas @ H[nbrs]
    return out, alphas

W_gat = rng.normal(0, 0.1, (d, 6))
a_vec = rng.normal(0, 0.1, 12)
out_gat, alphas_0 = gat_forward(X, A, W_gat, a_vec)
print('GAT 输出形状:', out_gat.shape)
print('节点 0 的邻居注意力权重（前 6 个邻居）:')
for j, a in zip(list(G.neighbors(0))[:6], alphas_0[:6]):
    print(f'  节点 {j}: α = {a:.3f}')
assert np.allclose(alphas_0.sum(), 1)
print('✅ 注意力系数归一化和为 1（softmax over 邻居）')

GAT 输出形状: (34, 6)
节点 0 的邻居注意力权重（前 6 个邻居）:
  节点 1: α = 0.063
  节点 2: α = 0.057
  节点 3: α = 0.058
  节点 4: α = 0.054
  节点 5: α = 0.060
  节点 6: α = 0.058
✅ 注意力系数归一化和为 1（softmax over 邻居）


## 3️⃣ GAT vs GCN 权重对比（可视化）

In [4]:
# 对比：GCN 平均权重 vs GAT 学到的注意力权重
nbrs0 = list(G.neighbors(0))
gcn_w = np.full(len(nbrs0), 1 / len(nbrs0))
gat_w = alphas_0[:len(nbrs0)]

plt.figure(figsize=(7, 4))
x = np.arange(len(nbrs0))
plt.bar(x - 0.2, gcn_w, width=0.4, label='GCN（等权平均）', color='#94a3b8')
plt.bar(x + 0.2, gat_w, width=0.4, label='GAT（注意力加权）', color='#f59e0b')
plt.xticks(x, [str(j) for j in nbrs0]); plt.ylabel('权重'); plt.legend()
plt.title('节点 0 的邻居聚合权重：GCN vs GAT')
plt.show()
print('➡️ GAT 的权重由"节点表示相关度"决定，可学到重要邻居')

➡️ GAT 的权重由"节点表示相关度"决定，可学到重要邻居


C:\Users\24260\AppData\Local\Temp\ipykernel_38976\1752819056.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ GraphSAGE：邻居采样 + 聚合

In [5]:
def sample_neighbors(G, node, k):
    nbrs = list(G.neighbors(node))
    if len(nbrs) <= k:
        return nbrs
    return list(rng.choice(nbrs, size=k, replace=False))

def sage_aggregate(X, G, sample_k=5):
    # 简化：采样邻居 -> 取邻居特征均值 -> 拼接自身特征（concat+linear 略）
    out = np.zeros_like(X)
    for i in range(len(nodes)):
        nb = sample_neighbors(G, nodes[i], sample_k)
        if nb:
            out[i] = np.concatenate([X[i], X[nb].mean(axis=0)])[:d] if False else                      0.5 * X[i] + 0.5 * X[nb].mean(axis=0)
        else:
            out[i] = X[i]
    return out

X_sage = sage_aggregate(X, G)
print('GraphSAGE 输出形状:', X_sage.shape)
print('节点 0 的采样邻居:', sample_neighbors(G, nodes[0], 5))
print('➡️ GraphSAGE 亮点：固定采样 K 个邻居 -> 计算量与图规模解耦，可处理新节点')

GraphSAGE 输出形状:

 (34, 4)
节点 0 的采样邻居: [np.int64(2), np.int64(31), np.int64(21), np.int64(11), np.int64(1)]
➡️ GraphSAGE 亮点：固定采样 K 个邻居 -> 计算量与图规模解耦，可处理新节点


## 5️⃣ 聚合函数对比（面试速答）

In [6]:
print('''GraphSAGE 三种聚合器：
  Mean  ：邻居均值（最快、基线）；
  LSTM  ：邻居按序过 LSTM（表达能力好但慢、需顺序）；
  Pool  ：逐维 max/mean 池化（对 permutation 不变、信息保留好）。

其他设计：
  多层：每层采样+聚合+非线性，可做归纳学习（inductive，新节点直接推理）；
  GCN 是直推式（transductive），GraphSAGE 是归纳式。
''')

GraphSAGE 三种聚合器：
  Mean  ：邻居均值（最快、基线）；
  LSTM  ：邻居按序过 LSTM（表达能力好但慢、需顺序）；
  Pool  ：逐维 max/mean 池化（对 permutation 不变、信息保留好）。

其他设计：
  多层：每层采样+聚合+非线性，可做归纳学习（inductive，新节点直接推理）；
  GCN 是直推式（transductive），GraphSAGE 是归纳式。



## 6️⃣ 应用场景与选择（面试速答）

In [7]:
print('''怎么选：
  小图/全监督/结构重要     -> GCN（简单稳定）
  邻居重要性差异大         -> GAT（注意力加权）
  大图/动态新增节点/冷启动 -> GraphSAGE（采样+归纳）
  图分类/同构判别          -> GIN（WL-test 增强）

典型任务：节点分类（反欺诈/用户标签）、边预测（推荐/链接预测）、
          图分类（分子性质）、社区发现（聚类）。
''')

怎么选：
  小图/全监督/结构重要     -> GCN（简单稳定）
  邻居重要性差异大         -> GAT（注意力加权）
  大图/动态新增节点/冷启动 -> GraphSAGE（采样+归纳）
  图分类/同构判别          -> GIN（WL-test 增强）

典型任务：节点分类（反欺诈/用户标签）、边预测（推荐/链接预测）、
          图分类（分子性质）、社区发现（聚类）。



## 📝 自测清单

- [ ] 能写出 GAT 注意力系数公式
- [ ] 能手写单头 GAT 前向并验证权重归一化
- [ ] 能讲 GraphSAGE 采样与聚合流程
- [ ] 能对比 GCN/GAT/GraphSAGE 的选择依据
- [ ] 能举出三大典型任务（节点/边/图分类）